In [3]:
using LowLevelFEM
using Tensors

In [4]:
function mergeFields(uu::Vector{<:LowLevelFEM.AbstractField})
    println("@@@@")
    nn = length(uu)
    nn ≥ 1 || error("mergeFields: empty input vector.")

    u0 = uu[1]
    T = typeof(u0)

    nodal = isNodal(u0)
    elemwise = isElementwise(u0)

    nodal || elemwise || error("mergeFields: invalid field storage.")

    for u in uu
        typeof(u) == T ||
            error("mergeFields: all fields must have the same concrete type.")
        u.model === u0.model ||
            error("mergeFields: all fields must belong to the same model.")
        u.type == u0.type ||
            error("mergeFields: all fields must have the same field type.")
        isNodal(u) == nodal ||
            error("mergeFields: cannot mix nodal and elementwise fields.")
    end

    n = sum(u.nsteps for u in uu)
    t = zeros(n)

    function shifted_time!(t, c, ui, prev)
        ns = ui.nsteps

        if c == 0
            if ns == 1
                t[1] = 1.0
            else
                t[1:ns] .= ui.t
            end
            return
        end

        dt = prev.nsteps > 1 ? prev.t[end] - prev.t[end-1] : 1.0

        if ns == 1
            t[c+1] = t[c] + dt
        else
            t[c+1:c+ns] .= (ui.t .- ui.t[1]) .+ (t[c] + dt)
        end
    end

    if nodal
        ndofs = size(u0.a, 1)
        a = zeros(ndofs, n)

        c = 0
        for i in 1:nn
            ui = uu[i]
            size(ui.a, 1) == ndofs ||
                error("mergeFields: nodal dof mismatch.")

            ns = ui.nsteps
            a[:, c+1:c+ns] .= ui.a
            shifted_time!(t, c, ui, i == 1 ? ui : uu[i-1])
            c += ns
        end

        return T([], a, t, [], n, u0.type, u0.model)

    else
        numElem = copy(u0.numElem)
        ne = length(numElem)
        A = Vector{Matrix{Float64}}(undef, ne)

        index0 = Dict(e => k for (k, e) in enumerate(numElem))

        for u in uu
            Set(u.numElem) == Set(numElem) ||
                error("mergeFields: element sets must be identical.")
        end

        for (ie, elem) in enumerate(numElem)
            rows = size(u0.A[ie], 1)
            A[ie] = zeros(rows, n)
        end

        c = 0
        for i in 1:nn
            ui = uu[i]
            idx = Dict(e => k for (k, e) in enumerate(ui.numElem))

            ns = ui.nsteps
            for elem in numElem
                ie0 = index0[elem]
                iei = idx[elem]

                size(ui.A[iei], 1) == size(A[ie0], 1) ||
                    error("mergeFields: element $elem has incompatible local size.")

                A[ie0][:, c+1:c+ns] .= ui.A[iei]
            end

            shifted_time!(t, c, ui, i == 1 ? ui : uu[i-1])
            c += ns
        end

        return T(A, [;;], t, numElem, n, u0.type, u0.model)
    end
end

mergeFields (generic function with 1 method)

In [5]:
structured_rect_mesh()

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)

In [6]:
p = (μ=mat.μ, λ=mat.λ)

ψ(C, p) = p.μ / 2 * (tr(C) - 3) - p.μ * log(sqrt(det(C))) +
          p.λ / 2 * log(sqrt(det(C)))^2

ψ (generic function with 1 method)

In [7]:
R = [
    1 0 0
    0 1 0
    0 0 0
    0 0 1
    0 0 0
    0 0 0
]

6×3 Matrix{Int64}:
 1  0  0
 0  1  0
 0  0  0
 0  0  1
 0  0  0
 0  0  0

In [8]:
F = TensorField(Pu, "body", [1 0 0; 0 1 0; 0 0 1])
C = F' * F

elementwise TensorField
[[1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;]  …  [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;], [1.0; 0.0; … ; 0.0; 1.0;;]]

In [9]:
D = tangentMatrix(ψ, C, params=p)

6×6 Matrix{ScalarField}:
 ScalarField([[2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;]  …  [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;]

In [10]:
S = IIPiolaKirchhoff(ψ, C, params=p)

elementwise TensorField
[[0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;]  …  [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;]]

In [11]:
D2 = R' * D * R

3×3 Matrix{ScalarField}:
 ScalarField([[2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;]  …  [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;], [2.69231e5; 2.69231e5; 2.69231e5; 2.69231e5;;]

In [12]:
S2 = R' * tensorToVector(S)

3-element Vector{ScalarField}:
 ScalarField([[0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;]  …  [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54  …  135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 1, :scalar, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f))
 ScalarField([[0.0; 0.0; 0.0; 0

In [13]:
R2 = [1 0; 0 1; 0 0]

3×2 Matrix{Int64}:
 1  0
 0  1
 0  0

In [14]:
R2' * tensorToMatrix(S) * R2

2×2 Matrix{ScalarField}:
 ScalarField([[0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;]  …  [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54  …  135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 1, :scalar, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 1.15385e5, 76923.1, 1.66667e5, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f))  …  ScalarField([[0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0

In [15]:
Kgeo = ∫(Grad(Pu) ⋅ R2' ⋅ tensorToMatrix(S) ⋅ R2 ⋅ Grad(Pu); Ω="body")

LoadError: AssertionError: size(A1, 1) == out_s

In [ ]:
Kgeo = ∫(Grad(Pu) ⋅ 1 ⋅ Grad(Pu); Ω="body")

sparse([1, 9, 79, 81, 2, 10, 80, 82, 3, 25  …  241, 6, 42, 44, 46, 48, 222, 224, 240, 242], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  241, 242, 242, 242, 242, 242, 242, 242, 242, 242], [0.6666666666666665, -0.16666666666666655, -0.16666666666666674, -0.33333333333333337, 0.6666666666666665, -0.16666666666666655, -0.16666666666666674, -0.33333333333333337, 0.6666666666666666, -0.16666666666666663  …  2.6666666666666665, -0.3333333333333335, -0.3333333333333336, -0.33333333333333315, -0.33333333333333337, -0.33333333333333326, -0.33333333333333315, -0.3333333333333331, -0.3333333333333336, 2.6666666666666665], 242, 242)

In [ ]:
Kmat = ∫(Grad(Pu) ⋅ R' ⋅ D ⋅ R ⋅ Grad(Pu); Ω="body")

LoadError: AssertionError: size(A1, 1) == out_s